In [1]:
%pip install fplstat

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
import pandas as pd
import numpy as np
from collections import defaultdict
import aiohttp
import asyncio
from fpl import FPL

bootstrap = requests.get(
    "https://fantasy.premierleague.com/api/bootstrap-static/"
).json()

fixtures = requests.get(
    "https://fantasy.premierleague.com/api/fixtures/"
).json()

players = bootstrap["elements"]
teams = bootstrap["teams"]

print(f"{len(players)} players loaded")
print(f"{len(fixtures)} fixtures loaded")

564 players loaded
380 fixtures loaded


In [3]:
players_df = pd.DataFrame(players)

players_df = players_df[
    [
        "id",
        "web_name",
        "team",
        "element_type",
        "now_cost",
        "total_points",
        "form",
        "minutes"
    ]
].copy()

players_df.columns = [
    "id",
    "name",
    "team",
    "position",
    "price",
    "points",
    "form",
    "minutes"
]

players_df["price"] = players_df["price"] / 10
players_df["form"] = players_df["form"].astype(float)

players_df.head()

,id,name,team,position,price,points,form,minutes
0,1,Raya,1,1,6.0,162,0.0,3330
1,2,Arrizabalaga,1,1,5.0,2,0.0,90
2,3,Meslier,1,1,5.0,0,0.0,0
3,4,Gabriel,1,2,8.0,209,0.0,2750
4,5,J.Timber,1,2,6.5,149,0.0,2452


In [4]:
fixture_weights = {
    1: 1.30,
    2: 1.15,
    3: 1.00,
    4: 0.90,
    5: 0.75
}

In [5]:
from collections import defaultdict

LOOKAHEAD = 5

team_fixtures = defaultdict(list)

for fixture in fixtures:

    if fixture["finished"]:
        continue

    home_team = fixture["team_h"]
    away_team = fixture["team_a"]

    home_diff = fixture["team_h_difficulty"]
    away_diff = fixture["team_a_difficulty"]

    team_fixtures[home_team].append(home_diff)
    team_fixtures[away_team].append(away_diff)

In [6]:
fixture_scores = {}

for team, diffs in team_fixtures.items():

    next_five = diffs[:LOOKAHEAD]

    score = sum(
        fixture_weights[d]
        for d in next_five
    )

    fixture_scores[team] = score

In [7]:
if len(fixture_scores) == 0:
    players_df["fixture_score"] = 1.0
else:
    players_df["fixture_score"] = (
        players_df["team"]
        .map(fixture_scores)
        .fillna(1.0)
    )

In [8]:
#LOAD HISTORICAL TRAINING DATA


import pandas as pd

print("Loading historical FPL data...")

# Historical gameweek-by-gameweek data
history_url = (
    "https://raw.githubusercontent.com/"
    "vaastav/Fantasy-Premier-League/"
    "master/data/2024-25/gws/merged_gw.csv"
)

history_df = pd.read_csv(history_url)

print(f"Loaded {len(history_df)} historical player-gameweeks.")

print("\nDataset shape:")
print(history_df.shape)

print("\nColumns:")
print(history_df.columns.tolist())

print("\nFirst five rows:")
display(history_df.head())

Loading historical FPL data...
Loaded 27605 historical player-gameweeks.

Dataset shape:
(27605, 49)

Columns:
['name', 'position', 'team', 'xP', 'assists', 'bonus', 'bps', 'clean_sheets', 'creativity', 'element', 'expected_assists', 'expected_goal_involvements', 'expected_goals', 'expected_goals_conceded', 'fixture', 'goals_conceded', 'goals_scored', 'ict_index', 'influence', 'kickoff_time', 'minutes', 'mng_clean_sheets', 'mng_draw', 'mng_goals_scored', 'mng_loss', 'mng_underdog_draw', 'mng_underdog_win', 'mng_win', 'modified', 'opponent_team', 'own_goals', 'penalties_missed', 'penalties_saved', 'red_cards', 'round', 'saves', 'selected', 'starts', 'team_a_score', 'team_h_score', 'threat', 'total_points', 'transfers_balance', 'transfers_in', 'transfers_out', 'value', 'was_home', 'yellow_cards', 'GW']

First five rows:


,name,position,team,xP,assists,bonus,bps,clean_sheets,creativity,element,...,team_h_score,threat,total_points,transfers_balance,transfers_in,transfers_out,value,was_home,yellow_cards,GW
0,Alex Scott,MID,Bournemouth,1.6,0,0,11,0,12.8,77,...,1,0.0,2,0,0,0,50,False,0,1
1,Carlos Miguel dos Santos Pereira,GK,Nott'm Forest,2.2,0,0,0,0,0.0,427,...,1,0.0,0,0,0,0,45,True,0,1
2,Tomiyasu Takehiro,DEF,Arsenal,0.0,0,0,0,0,0.0,22,...,2,0.0,0,0,0,0,50,True,0,1
3,Malcolm Ebiowei,MID,Crystal Palace,0.0,0,0,0,0,0.0,197,...,2,0.0,0,0,0,0,45,False,0,1
4,Ben Brereton Díaz,MID,Southampton,1.0,0,0,-2,0,14.0,584,...,1,16.0,1,0,0,0,55,False,1,1


In [9]:
#FEATURE ENGINEERING


from sklearn.preprocessing import LabelEncoder

print("Preparing machine learning dataset...")

# Selects only features available
# in both history_df and players_df


ml_df = history_df[
    [
        "minutes",
        "value",
        "position",
        "team",
        "total_points"
    ]
].copy()


# Converts numeric columns


ml_df["minutes"] = pd.to_numeric(
    ml_df["minutes"],
    errors="coerce"
)

ml_df["value"] = pd.to_numeric(
    ml_df["value"],
    errors="coerce"
)


# Encodes categorical variables

position_encoder = LabelEncoder()
team_encoder = LabelEncoder()

ml_df["position"] = position_encoder.fit_transform(
    ml_df["position"]
)

ml_df["team"] = team_encoder.fit_transform(
    ml_df["team"]
)


# Removes incomplete rows


ml_df = ml_df.dropna()

print(f"Training rows: {len(ml_df)}")

# Features and target


feature_columns = [
    "minutes",
    "value",
    "position",
    "team"
]

X = ml_df[feature_columns]

y = ml_df["total_points"]

print("\nFeatures ready!")
print(f"{X.shape[1]} features")
print(f"{X.shape[0]} training examples")

print("\nFeatures used:")
for feature in feature_columns:
    print("-", feature)

Preparing machine learning dataset...
Training rows: 27605

Features ready!
4 features
27605 training examples

Features used:
- minutes
- value
- position
- team


In [10]:
#Trains a Random Forest model

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Training Random Forest model...")


# Train / Test Split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# Build Model


rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

# Train


rf_model.fit(X_train, y_train)

print("Training complete!")


# Evaluate


predictions = rf_model.predict(X_test)

mae = mean_absolute_error(
    y_test,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predictions
    )
)

r2 = r2_score(
    y_test,
    predictions
)

print("\nMODEL PERFORMANCE")
print(f"Mean Absolute Error : {mae:.2f}")
print(f"Root Mean Squared Error : {rmse:.2f}")
print(f"R² Score : {r2:.3f}")


# Feature Importance


importance = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": rf_model.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print("\n FEATURE IMPORTANCE")
display(importance)

print("\nModel is ready for live FPL predictions.")

Training Random Forest model...
Training complete!

MODEL PERFORMANCE
Mean Absolute Error : 0.78
Root Mean Squared Error : 1.80
R² Score : 0.458

 FEATURE IMPORTANCE


,Feature,Importance
0,minutes,0.717515
1,value,0.173945
2,position,0.056917
3,team,0.051623



Model is ready for live FPL predictions.


In [11]:
#ml setup inspection


import pandas as pd

print("HISTORY DATA INFO")

# 1. Columns in historical dataset
print("\n[1] history_df columns:")
print(history_df.columns.tolist())

# 2. First row preview (helps identify feature names + structure)
print("\n[2] history_df sample row:")
print(history_df.head(1).T)

# 3. Check for common ML-relevant columns
print("\n[3] Key column presence check:")

key_columns = [
    "total_points",
    "minutes",
    "bps",
    "influence",
    "creativity",
    "threat",
    "ict_index",
    "opponent_team",
    "was_home",
    "team_h",
    "team_a",
    "expected_goals",
    "expected_assists"
]

for col in key_columns:
    print(f"{col}: {'YES' if col in history_df.columns else 'NO'}")

print("\nDONE")

HISTORY DATA INFO

[1] history_df columns:
['name', 'position', 'team', 'xP', 'assists', 'bonus', 'bps', 'clean_sheets', 'creativity', 'element', 'expected_assists', 'expected_goal_involvements', 'expected_goals', 'expected_goals_conceded', 'fixture', 'goals_conceded', 'goals_scored', 'ict_index', 'influence', 'kickoff_time', 'minutes', 'mng_clean_sheets', 'mng_draw', 'mng_goals_scored', 'mng_loss', 'mng_underdog_draw', 'mng_underdog_win', 'mng_win', 'modified', 'opponent_team', 'own_goals', 'penalties_missed', 'penalties_saved', 'red_cards', 'round', 'saves', 'selected', 'starts', 'team_a_score', 'team_h_score', 'threat', 'total_points', 'transfers_balance', 'transfers_in', 'transfers_out', 'value', 'was_home', 'yellow_cards', 'GW']

[2] history_df sample row:
                                               0
name                                  Alex Scott
position                                     MID
team                                 Bournemouth
xP                              

In [12]:
#predicting player quality

print("Predicting player quality...")

from sklearn.preprocessing import LabelEncoder

# Encode live data the same way as training


live_df = players_df.copy()

# Position encoding
position_encoder = LabelEncoder()

live_df["position"] = position_encoder.fit_transform(
    live_df["position"]
)

# Team encoding
team_encoder = LabelEncoder()

live_df["team"] = team_encoder.fit_transform(
    live_df["team"]
)


# Build prediction dataframe


prediction_features = pd.DataFrame()

prediction_features["minutes"] = live_df["minutes"]

# Historical value column is price × 10
prediction_features["value"] = live_df["price"] * 10

prediction_features["position"] = live_df["position"]

prediction_features["team"] = live_df["team"]

prediction_features = prediction_features.fillna(0)


# Predict base player quality


players_df["ml_prediction"] = rf_model.predict(
    prediction_features[feature_columns]
)


# Scale using recent form


players_df["ml_prediction"] = (
    players_df["ml_prediction"]
    *
    (
        0.5
        +
        players_df["form"] /
        (players_df["form"].max() + 0.01)
    )
)


# Apply next-five fixture adjustment


players_df["projected_points"] = (
    players_df["ml_prediction"]
    *
    players_df["fixture_score"]
)

# Value metric


players_df["value_score"] = (
    players_df["projected_points"]
    /
    players_df["price"]
)

print("\nML predictions complete.")

print("\nTop 10 projected players:")

display(
    players_df[
        [
            "name",
            "price",
            "form",
            "fixture_score",
            "ml_prediction",
            "projected_points",
            "value_score"
        ]
    ]
    .sort_values(
        "projected_points",
        ascending=False
    )
    .head(10)
)

Predicting player quality...

ML predictions complete.

Top 10 projected players:


,name,price,form,fixture_score,ml_prediction,projected_points,value_score
435,B.Fernandes,12.0,0.0,5.20,4.509271,23.448209,1.954017
383,Isak,9.0,0.0,5.30,2.839925,15.051602,1.672400
402,Semenyo,8.5,0.0,5.20,2.826438,14.697480,1.729115
94,O.Dango,6.5,0.0,5.05,2.859059,14.438250,2.221269
139,Rogers,7.5,0.0,5.05,2.657744,13.421607,1.789548
163,João Pedro,7.5,0.0,5.05,2.657744,13.421607,1.789548
437,Cunha,8.0,0.0,5.20,2.569362,13.360683,1.670085
436,Mbeumo,8.0,0.0,5.20,2.569362,13.360683,1.670085
489,Gibbs-White,8.0,0.0,5.10,2.617883,13.351203,1.668900
104,Thiago,8.0,0.0,5.05,2.597215,13.115934,1.639492


In [13]:
print(fixtures[:2])

[{'code': 2645195, 'event': 1, 'finished': False, 'finished_provisional': False, 'id': 1, 'kickoff_time': '2026-08-21T19:00:00Z', 'minutes': 0, 'provisional_start_time': False, 'started': False, 'team_a': 7, 'team_a_score': None, 'team_h': 1, 'team_h_score': None, 'stats': [], 'team_h_difficulty': 2, 'team_a_difficulty': 5, 'pulse_id': 0}, {'code': 2645198, 'event': 1, 'finished': False, 'finished_provisional': False, 'id': 4, 'kickoff_time': '2026-08-22T11:30:00Z', 'minutes': 0, 'provisional_start_time': False, 'started': False, 'team_a': 16, 'team_a_score': None, 'team_h': 11, 'team_h_score': None, 'stats': [], 'team_h_difficulty': 4, 'team_a_difficulty': 2, 'pulse_id': 0}]


In [14]:
players_df["value_score"] = (
    players_df["projected_points"]
    / players_df["price"]
)

In [ ]:
# Enter FPL Team ID and fetch squad


team_id = int(input("Enter your FPL Team ID: "))

current_gw = next(
    event["id"]
    for event in bootstrap["events"]
    if event["is_current"]
)

print(current_gw)

picks_url = (
    f"https://fantasy.premierleague.com/api/"
    f"entry/{team_id}/event/{current_gw}/picks/"
)

picks_data = requests.get(picks_url).json()

user_squad = []

for pick in picks_data["picks"]:
    user_squad.append({
        "id": pick["element"],
        "selling_price": pick["selling_price"] / 10
    })

user_squad_df = pd.DataFrame(user_squad)
user_squad_ids = user_squad_df["id"].tolist()

bank = (
    picks_data["entry_history"]["bank"] / 10
)

print(f"Bank: £{bank}m")

In [ ]:
squad_df = (
    players_df[
        players_df["id"].isin(user_squad_ids)
    ]
    .merge(
        user_squad_df,
        on="id",
        how="left"
    )
)

In [ ]:
squad_df["projected_points"] = squad_df["projected_points"].replace(0,np.nan)

In [ ]:
squad_df["sell_score"] = (
    squad_df["price"]
    / (squad_df["projected_points"] + 0.001)
)

In [ ]:
print(players_df["form"].describe())

In [ ]:
print(squad_df[["name", "projected_points", "sell_score"]].head())

In [ ]:
player_out = squad_df.loc[
    squad_df["sell_score"].idxmax()
]

player_out

In [ ]:
max_price = (
    player_out["selling_price"]
    + bank
)

print(
    f"Sell {player_out['name']} "
    f"for up to £{max_price:.1f}m"
)

In [ ]:
owned_players = set(
    squad_df["id"]
)

In [ ]:
candidates = players_df[
    (players_df["position"] == player_out["position"])
    &
    (players_df["price"] <= max_price)
    &
    (~players_df["id"].isin(owned_players))
].copy()

In [ ]:
candidates["gain"] = (
    candidates["projected_points"]
    -
    player_out["projected_points"]
)

In [ ]:
candidates["value_score"] = (
    candidates["projected_points"]
    /
    candidates["price"]
)

In [ ]:
candidates["budget_usage"] = (
    candidates["price"]
    /
    max_price
)

In [ ]:
candidates["recommendation_score"] = (
    (
        0.5 * candidates["value_score"]
        +
        0.3 * candidates["fixture_score"]
        +
        0.2 * candidates["budget_usage"]
    )
    *
    np.maximum(
        candidates["gain"],
        0
    )
)

In [ ]:
recommendations = candidates.sort_values(
    "recommendation_score",
    ascending=False
)

recommendations[
    [
        "name",
        "price",
        "projected_points",
        "fixture_score",
        "gain",
        "recommendation_score"
    ]
].head(10) #end of transfer recommendation code

In [ ]:
# best triple captain feature
remaining_fixtures = [
    f for f in fixtures
    if not f["finished"]
]

triple_scores = []

for _, player in players_df.iterrows():

    gw_scores = defaultdict(float)

    for fixture in remaining_fixtures:

        gw = fixture["event"]

        if gw is None:
            continue

        if fixture["team_h"] == player["team"]:
            difficulty = fixture["team_h_difficulty"]

        elif fixture["team_a"] == player["team"]:
            difficulty = fixture["team_a_difficulty"]

        else:
            continue

        fixture_projection = (
            player["projected_points"] *
            fixture_weights[difficulty]
)

        gw_scores[gw] += fixture_projection

    if len(gw_scores) == 0:
        continue

    first_half = {
        gw: score
        for gw, score in gw_scores.items()
        if gw <= 19
    }

    second_half = {
        gw: score
        for gw, score in gw_scores.items()
        if gw >= 20
    }

    if first_half:

        best_gw = max(first_half, key=first_half.get)

        triple_scores.append({
            "id": player["id"],
            "name": player["name"],
            "half": 1,
            "gw": best_gw,
            "expected_points": first_half[best_gw]
        })

    if second_half:

        best_gw = max(second_half, key=second_half.get)

        triple_scores.append({
            "id": player["id"],
            "name": player["name"],
            "half": 2,
            "gw": best_gw,
            "expected_points": second_half[best_gw]
        })

triple_df = pd.DataFrame(triple_scores)

In [ ]:
for half in [1, 2]:

    best = (
        triple_df[triple_df["half"] == half]
        .sort_values("expected_points", ascending=False)
        .iloc[0]
    )

    print(f"\nBest triple captain in game: (Half {half})")
    print(f"Player: {best['name']}")
    print(f"Gameweek: {best['gw']}")
    print(f"Expected Points: {best['expected_points']:.2f}")

In [ ]:
user_tc = triple_df[
    triple_df["id"].isin(user_squad_ids)
]

for half in [1, 2]:

    best = (
        user_tc[user_tc["half"] == half]
        .sort_values("expected_points", ascending=False)
        .iloc[0]
    )

    print(f"\nYour best triple captain (Half {half})")
    print(f"Player: {best['name']}")
    print(f"Gameweek: {best['gw']}")
    print(f"Expected Points: {best['expected_points']:.2f}")

In [ ]:
#best free hit week feature

from collections import defaultdict

#finds the teams weakest gw

remaining_fixtures = [
    f for f in fixtures
    if not f["finished"] and f["event"] is not None
]

# key = gameweek
# value = projected team score
team_gw_scores = defaultdict(float)

for _, player in squad_df.iterrows():

    player_team = player["team"]

    for fixture in remaining_fixtures:

        if fixture["team_h"] == player_team:

            difficulty = fixture["team_h_difficulty"]

        elif fixture["team_a"] == player_team:

            difficulty = fixture["team_a_difficulty"]

        else:
            continue

        gw = fixture["event"]

        # Projected points for THIS fixture only
        fixture_projection = (
            player["projected_points"]
            *
            fixture_weights[difficulty]
        )

        # Automatically adds both fixtures if
        # the team has a double gameweek
        team_gw_scores[gw] += fixture_projection



# Split season into halves

first_half_scores = {
    gw: score
    for gw, score in team_gw_scores.items()
    if gw <= 19
}

second_half_scores = {
    gw: score
    for gw, score in team_gw_scores.items()
    if gw >= 20
}


# Find weakest gameweek

worst_first_half = min(
    first_half_scores,
    key=first_half_scores.get
)

worst_second_half = min(
    second_half_scores,
    key=second_half_scores.get
)


print("Free hit recommendation:")

print(f"\nFirst Half")
print(f"Recommended Gameweek: {worst_first_half}")
print(f"Projected Team Points: {first_half_scores[worst_first_half]:.2f}")

print(f"\nSecond Half")
print(f"Recommended Gameweek: {worst_second_half}")
print(f"Projected Team Points: {second_half_scores[worst_second_half]:.2f}")

In [ ]:
squad_value = squad_df["selling_price"].sum()

free_hit_budget = squad_value + bank

print("Free hit budget:")
print(f"Squad Selling Value: £{squad_value:.1f}m")
print(f"Money In Bank: £{bank:.1f}m")
print(f"Available Budget: £{free_hit_budget:.1f}m")

In [ ]:
!pip install pulp

In [ ]:
import pulp


# SELECT GAMEWEEK (we'll use worst first half as default)

target_gw = worst_first_half   # change to worst_second_half later


# Build player pool for that GW


gw_fixtures = [
    f for f in fixtures
    if f["event"] == target_gw and not f["finished"]
]

# Precompute player GW scores
player_gw_score = {}

for _, player in players_df.iterrows():

    team_id = player["team"]
    score = 0

    for fixture in gw_fixtures:

        if fixture["team_h"] == team_id:
            difficulty = fixture["team_h_difficulty"]

        elif fixture["team_a"] == team_id:
            difficulty = fixture["team_a_difficulty"]

        else:
            continue

        score += (
            player["projected_points"] *
            fixture_weights[difficulty]
        )

    player_gw_score[player["id"]] = score



#model

model = pulp.LpProblem("Free_Hit_Optimisation", pulp.LpMaximize)

players = players_df["id"].tolist()

# decision variables: 1 if player is selected
x = pulp.LpVariable.dicts("player", players, cat="Binary")


#objective

model += pulp.lpSum([
    player_gw_score.get(i, 0) * x[i]
    for i in players
])



#constraints


# Budget constraint
model += pulp.lpSum([
    players_df.loc[players_df["id"] == i, "price"].values[0] * x[i]
    for i in players
]) <= free_hit_budget


# Squad size = 15
model += pulp.lpSum([x[i] for i in players]) == 15


# Position constraints
model += pulp.lpSum([
    x[i] for i in players
    if players_df.loc[players_df["id"] == i, "position"].values[0] == 1
]) == 2  # GKs

model += pulp.lpSum([
    x[i] for i in players
    if players_df.loc[players_df["id"] == i, "position"].values[0] == 2
]) == 5  # DEF

model += pulp.lpSum([
    x[i] for i in players
    if players_df.loc[players_df["id"] == i, "position"].values[0] == 3
]) == 5  # MID

model += pulp.lpSum([
    x[i] for i in players
    if players_df.loc[players_df["id"] == i, "position"].values[0] == 4
]) == 3  # FWD


# Max 3 players per team
for team in players_df["team"].unique():

    model += pulp.lpSum([
        x[i]
        for i in players
        if players_df.loc[players_df["id"] == i, "team"].values[0] == team
    ]) <= 3



#solve

model.solve()

print("Status:", pulp.LpStatus[model.status])



#output squad

optimal_team = players_df[
    [x[i].value() == 1 for i in players]
].copy()

print("\nOPTIMAL FREE HIT TEAM:")
print(optimal_team[["name", "price", "position", "projected_points"]])
print("\nTotal projected score:",
      sum(player_gw_score.get(i, 0) for i in players if x[i].value() == 1))

In [ ]:
#extracts optimal team


selected_ids = [i for i in players if x[i].value() == 1]

optimal_team = players_df[
    players_df["id"].isin(selected_ids)
].copy()


#team summary


total_optimal_points = sum(
    player_gw_score.get(i, 0)
    for i in selected_ids
)

current_team_points = sum(
    squad_gw_scores.get(i, 0)
    for i in squad_df["id"]
) if 'squad_gw_scores' in globals() else None


print("OPTIMAL FREE HIT TEAM:\n")

print(optimal_team[
    ["name", "position", "price", "projected_points"]
].sort_values("position"))



print(f"Total Optimal Projected Points: {total_optimal_points:.2f}")


#optimal comparison

if current_team_points is not None:
    print(f"Current Squad Projected Points: {current_team_points:.2f}")
    print(f"Expected Gain from Free Hit: {total_optimal_points - current_team_points:.2f}")

In [ ]:

#free hit(second half of season)
#worst_second_half gw is best time to use free hit
target_gw = worst_second_half


#build gw fixtures


gw_fixtures = [
    f for f in fixtures
    if f["event"] == target_gw and not f["finished"]
]



#player gw scores

player_gw_score = {}

for _, player in players_df.iterrows():

    team_id = player["team"]
    score = 0

    for fixture in gw_fixtures:

        if fixture["team_h"] == team_id:
            difficulty = fixture["team_h_difficulty"]

        elif fixture["team_a"] == team_id:
            difficulty = fixture["team_a_difficulty"]

        else:
            continue

        score += (
            player["projected_points"] *
            fixture_weights[difficulty]
        )

    player_gw_score[player["id"]] = score



#solves model again


model2 = pulp.LpProblem("Free_Hit_Second_Half", pulp.LpMaximize)

players = players_df["id"].tolist()

x2 = pulp.LpVariable.dicts("player", players, cat="Binary")


#objective
model2 += pulp.lpSum([
    player_gw_score.get(i, 0) * x2[i]
    for i in players
])


# budget
model2 += pulp.lpSum([
    players_df.loc[players_df["id"] == i, "price"].values[0] * x2[i]
    for i in players
]) <= free_hit_budget


#squad size
model2 += pulp.lpSum([x2[i] for i in players]) == 15


#position constraints
model2 += pulp.lpSum([
    x2[i] for i in players
    if players_df.loc[players_df["id"] == i, "position"].values[0] == 1
]) == 2

model2 += pulp.lpSum([
    x2[i] for i in players
    if players_df.loc[players_df["id"] == i, "position"].values[0] == 2
]) == 5

model2 += pulp.lpSum([
    x2[i] for i in players
    if players_df.loc[players_df["id"] == i, "position"].values[0] == 3
]) == 5

model2 += pulp.lpSum([
    x2[i] for i in players
    if players_df.loc[players_df["id"] == i, "position"].values[0] == 4
]) == 3


# only allows a max of 3 per team
for team in players_df["team"].unique():

    model2 += pulp.lpSum([
        x2[i]
        for i in players
        if players_df.loc[players_df["id"] == i, "team"].values[0] == team
    ]) <= 3


#solve
model2.solve()

print("Status:", pulp.LpStatus[model2.status])


#output


selected_ids = [i for i in players if x2[i].value() == 1]

optimal_team_2 = players_df[
    players_df["id"].isin(selected_ids)
].copy()

total_points_2 = sum(
    player_gw_score.get(i, 0)
    for i in selected_ids
)

print("\nOptimal free hit team(second half):\n")

print(optimal_team_2[
    ["name", "position", "price", "projected_points"]
].sort_values("position"))

print("\nTotal projected score:", total_points_2)

In [ ]:
from collections import defaultdict

#wildcard settings


LOOKAHEAD = 6

remaining_fixtures = [
    f for f in fixtures
    if (not f["finished"]) and (f["event"] is not None)
]

#project user's squad


team_gw_scores = defaultdict(float)

for _, player in squad_df.iterrows():

    team = player["team"]

    for fixture in remaining_fixtures:

        if fixture["team_h"] == team:
            difficulty = fixture["team_h_difficulty"]

        elif fixture["team_a"] == team:
            difficulty = fixture["team_a_difficulty"]

        else:
            continue

        gw = fixture["event"]

        projection = (
            player["projected_points"]
            *
            fixture_weights[difficulty]
        )

        # Double gameweeks are automatically included
        team_gw_scores[gw] += projection


# ROLLING 6-GW SCORES


rolling_scores = {}

all_gws = sorted(team_gw_scores.keys())

for start_gw in all_gws:

    total = 0

    for gw in range(start_gw, start_gw + LOOKAHEAD):

        total += team_gw_scores.get(gw, 0)

    rolling_scores[start_gw] = total


# We split in half the season as you get a wildcard in each half


first_half = {
    gw: score
    for gw, score in rolling_scores.items()
    if gw <= 19
}

second_half = {
    gw: score
    for gw, score in rolling_scores.items()
    if gw >= 20
}



# +best wildcard week


wildcard_first = min(
    first_half,
    key=first_half.get
)

wildcard_second = min(
    second_half,
    key=second_half.get
)


print("Wildcard recommendation:\n")

print("First half of the season:")
print(f"Wildcard in GW {wildcard_first}")
print(f"Projected next {LOOKAHEAD} GW points: {first_half[wildcard_first]:.2f}")

print()

print("Second half of the season:")
print(f"Wildcard in GW {wildcard_second}")
print(f"Projected next {LOOKAHEAD} GW points: {second_half[wildcard_second]:.2f}")

In [ ]:
#wildcard player optimisations


# Use the first-half Wildcard by default.
# Change to wildcard_second for the second-half optimisation.
target_gw = wildcard_first

LOOKAHEAD = 6

future_points = {}

for _, player in players_df.iterrows():

    team = player["team"]
    total_projection = 0

    for fixture in fixtures:

        if fixture["finished"]:
            continue

        if fixture["event"] is None:
            continue

        # Only consider fixtures in the Wildcard window
        if not (target_gw <= fixture["event"] < target_gw + LOOKAHEAD):
            continue

        if fixture["team_h"] == team:
            difficulty = fixture["team_h_difficulty"]

        elif fixture["team_a"] == team:
            difficulty = fixture["team_a_difficulty"]

        else:
            continue

        # Adds every fixture, including doubles
        total_projection += (
            player["projected_points"]
            * fixture_weights[difficulty]
        )

    future_points[player["id"]] = total_projection



# add to dataframe


players_df["future_points"] = (
    players_df["id"]
    .map(future_points)
    .fillna(0)
)

players_df["future_value"] = (
    players_df["future_points"]
    / players_df["price"]
)

print("Long term play projections:")

print(
    players_df[
        [
            "name",
            "price",
            "future_points",
            "future_value"
        ]
    ]
    .sort_values("future_points", ascending=False)
    .head(20)
)

In [ ]:
import pulp


# wildcard optimisation model


model = pulp.LpProblem(
    "Wildcard_Optimisation",
    pulp.LpMaximize
)

players = players_df["id"].tolist()

# Binary decision variables
x = pulp.LpVariable.dicts(
    "player",
    players,
    cat="Binary"
)


# objective function

# Maximise projected points over
# the next LOOKAHEAD gameweeks

model += pulp.lpSum(
    players_df.loc[
        players_df["id"] == i,
        "future_points"
    ].values[0] * x[i]
    for i in players
)


# budget constraint


model += pulp.lpSum(
    players_df.loc[
        players_df["id"] == i,
        "price"
    ].values[0] * x[i]
    for i in players
) <= free_hit_budget



# squad size


model += pulp.lpSum(
    x[i]
    for i in players
) == 15



# position constraints


# Goalkeepers
model += pulp.lpSum(
    x[i]
    for i in players
    if players_df.loc[
        players_df["id"] == i,
        "position"
    ].values[0] == 1
) == 2


# Defenders
model += pulp.lpSum(
    x[i]
    for i in players
    if players_df.loc[
        players_df["id"] == i,
        "position"
    ].values[0] == 2
) == 5


# Midfielders
model += pulp.lpSum(
    x[i]
    for i in players
    if players_df.loc[
        players_df["id"] == i,
        "position"
    ].values[0] == 3
) == 5


# Forwards
model += pulp.lpSum(
    x[i]
    for i in players
    if players_df.loc[
        players_df["id"] == i,
        "position"
    ].values[0] == 4
) == 3



# ensures maximum 3 players per club


for team in players_df["team"].unique():

    model += pulp.lpSum(
        x[i]
        for i in players
        if players_df.loc[
            players_df["id"] == i,
            "team"
        ].values[0] == team
    ) <= 3


#solve

model.solve()

print("Status:", pulp.LpStatus[model.status])

In [ ]:
# extract optimal wildcard team

selected_ids = [
    i for i in players
    if x[i].value() == 1
]

wildcard_team = players_df[
    players_df["id"].isin(selected_ids)
].copy()

print("OPTIMAL WILDCARD SQUAD:\n")

print(
    wildcard_team[
        [
            "name",
            "position",
            "price",
            "future_points",
            "future_value"
        ]
    ]
    .sort_values(["position", "future_points"],
                 ascending=[True, False])
)



# transfers out


players_out = squad_df[
    ~squad_df["id"].isin(selected_ids)
]


print("PLAYERS TO SELL:")


print(
    players_out[
        [
            "name",
            "position",
            "selling_price",
            "projected_points"
        ]
    ]
    .sort_values("position")
)


# transfers in

players_in = wildcard_team[
    ~wildcard_team["id"].isin(user_squad_ids)
]

print("PLAYERS TO BUY:")


print(
    players_in[
        [
            "name",
            "position",
            "price",
            "future_points"
        ]
    ]
    .sort_values("position")
)

In [ ]:
#wildcard summary

current_future_points = squad_df["future_points"].sum()

wildcard_future_points = wildcard_team["future_points"].sum()

improvement = (
    wildcard_future_points -
    current_future_points
)

budget_used = wildcard_team["price"].sum()

remaining_budget = wildcard_budget - budget_used


print("WILDCARD SUMMARY:")

print(f"Recommended Wildcard GW: {target_gw}")

print(f"\nCurrent Squad Projection : {current_future_points:.2f}")
print(f"Wildcard Squad Projection: {wildcard_future_points:.2f}")
print(f"Expected Improvement     : +{improvement:.2f} points")

print(f"\nBudget Available : £{wildcard_budget:.1f}m")
print(f"Budget Used      : £{budget_used:.1f}m")
print(f"Money Remaining  : £{remaining_budget:.1f}m")

print(f"\nPlayers Sold : {len(players_out)}")
print(f"Players Bought: {len(players_in)}")


#best captain

captain = wildcard_team.loc[
    wildcard_team["future_points"].idxmax()
]

print("BEST CAPTAIN AFTER WILDCARD:")


print(f"Player: {captain['name']}")
print(f"Projected Points: {captain['future_points']:.2f}")


#best value signing

best_value = wildcard_team.loc[
    wildcard_team["future_value"].idxmax()
]

print("BEST VALUE SIGNING:")

print(f"Player: {best_value['name']}")
print(f"Price: £{best_value['price']:.1f}m")
print(f"Future Value Score: {best_value['future_value']:.2f}")


#top 5 signings

print("TOP 5 LONG-TERM SIGNINGS")


print(
    wildcard_team[
        [
            "name",
            "price",
            "future_points",
            "future_value"
        ]
    ]
    .sort_values(
        "future_points",
        ascending=False
    )
    .head(5)
)

In [ ]:
#gaussian model

# Mean vector (expected points)

mu = wildcard_team["future_points"].values

# Estimate uncertainty
# (20% coefficient of variation)

sigma = 0.20 * mu

# Covariance matrix
# Assume players are independent for now

covariance = np.diag(sigma**2)

print("Mean Vector Shape:", mu.shape)
print("Covariance Matrix Shape:", covariance.shape)

In [ ]:
#monte carlo simulation

NUM_SIMULATIONS = 10000

samples = np.random.multivariate_normal(
    mean=mu,
    cov=covariance,
    size=NUM_SIMULATIONS
)

team_scores = samples.sum(axis=1)

print("Simulation Complete")

In [ ]:
#results

mean_score = np.mean(team_scores)

std_score = np.std(team_scores)

lower95 = np.percentile(team_scores,2.5)

upper95 = np.percentile(team_scores,97.5)

print("Gaussian validation:\n")

print(f"Expected Score : {mean_score:.2f}")

print(f"Standard Deviation : {std_score:.2f}")

print(f"95% Confidence Interval")

print(f"{lower95:.2f}  ->  {upper95:.2f}")

In [ ]:
#probability of improvement

current_score = squad_df["future_points"].sum()

probability = np.mean(team_scores > current_score)

print(f"Current Squad Projection : {current_score:.2f}")

print(f"Probability Wildcard Is Better : {probability*100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plt.hist(team_scores, bins=50)

plt.axvline(
    current_score,
    linestyle="--",
    label="Current Squad"
)

plt.axvline(
    mean_score,
    linestyle="-",
    label="Wildcard Mean"
)

plt.xlabel("Projected Points")

plt.ylabel("Frequency")

plt.title("Monte Carlo Validation of Wildcard")

plt.legend()

plt.show()